# Initial-shock variability: E3SM and CESM-SMYLE

Archive-backed companion to `1a_refactor_atm_leadtime_acc_skill_map.ipynb`.

**Configure → inventory and cache plan → monthly archive preparation and common-grid
calculation → cached diagnostics → case comparisons → individual initialization plots.**

This uses the same E3SM post-processing archive, CESM-SMYLE benchmark archive,
observation accessors and regridding utilities as `1a`. The calculation is separate:
no detrending, lead-drift removal, or ACC anomaly inputs. The scalar ratio is the
model ensemble-mean temporal standard deviation divided by observed standard deviation.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

repo_root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'esp_lab').is_dir()), None)
if repo_root is not None:
    sys.path.insert(0, str(repo_root))
from workflows.diagnostics.initial_shock_archive import plan_archive_run, compute_archive_plan
from esp_lab.diagnostics.initial_shock import plot_std_ratio
from esp_lab.utils.dask_util import DaskConfig, get_cluster_client
from esp_lab.utils.resource_utils import ResourceTracker


## User setup and workflow configuration

Select a field and the E3SM cases here, as in `1a`. Input files are discovered through
the existing archive accessors; no manually prepared monthly filenames are needed.
`TREFHT` is the temperature diagnostic used in the NCL example. All conversion
factors below assume the same archive conventions as `1a`; PRECT is mm/day after conversion.

The original NCL index needs five annual samples (60 months). With the current
24-month archive, these defaults explicitly compute an **exploratory two-block
ratio**, using two consecutive 12-month means. May blocks are May–April; November
blocks are November–October. This is not the original five-year statistic.


In [ ]:
VAR_CONFIG = {
    'TREFHT': dict(obs_product='ERA5', obs_variable='tas', units='degC',
                   model_scale=1., model_offset=-273.15, smyle_scale=1., smyle_offset=-273.15,
                   obs_scale=1., obs_offset=-273.15),
    'TS': dict(obs_product='ERA5', obs_variable='ts', units='degC',
               model_scale=1., model_offset=-273.15, smyle_scale=1., smyle_offset=-273.15,
               obs_scale=1., obs_offset=-273.15),
    'PRECT': dict(obs_product='GPCP_v2.3', obs_variable='PRECT', units='mm/day',
                  model_scale=86400000., model_offset=0., smyle_scale=86400000., smyle_offset=0.,
                  obs_scale=1., obs_offset=0.),
    'PSL': dict(obs_product='ERA5', obs_variable='psl', units='hPa',
                model_scale=.01, model_offset=0., smyle_scale=.01, smyle_offset=0.,
                obs_scale=.01, obs_offset=0.),
}
field = 'TREFHT'
variable = dict(VAR_CONFIG[field], field=field, model_variable=field, smyle_variable=field)

E3SM_CASES = {
    'E3SM-FOSIRL': dict(case_prefix='WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL', cache_tag='JRA55_FOSIRL'),
    'E3SM-Reanalysis': dict(case_prefix='WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce', cache_tag='Reanalysis'),
    'E3SM-4DEnVarOcn': dict(case_prefix='WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn', cache_tag='4DEnVarOcn'),
}
WORKFLOW_SETTINGS = {
    'paths': {
        's2d_diag_root': '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag',
        'figure_outdir': '/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag/initial_shock',
    },
    'run': {'years': [1980, 2011], 'init_months': [5, 11], 'nlead': 24, 'smoke_mode': False},
    'e3sm': {
        'data_dir': '/global/cfs/cdirs/e3sm/S2S2D/post_process',
        'nens': 10, 'grid': '180x360_aave', 'ts_split': '2yr', 'engine': 'netcdf4',
        'chunks': {'Y': 3, 'L': 24, 'M': 2, 'lat': 90, 'lon': 180},
    },
    'smyle': {
        'include': True, 'nens': 20,
        'benchmark_dir': '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE',
        'chunks': {'Y': 3, 'L': 24, 'M': 2, 'lat': 96, 'lon': 144},
    },
    'obs': {
        'data_dir': '/global/cfs/cdirs/e3sm/e3sm_diags/obs_for_e3sm_diags/time-series',
        'chunks': {'time': 24, 'lat': 90, 'lon': 180},
    },
    'regrid': {'target_dlat': 1., 'target_dlon': 1., 'method': 'conservative', 'periodic': True},
    'metric': {'window_months': 24, 'block_months': 12, 'start_lead': 0,
               'min_samples': None, 'min_area_fraction': .9},
    'cache': {'mode': 'auto'},  # auto / rebuild / require; inventory identity always checked
    'dask': {'enabled': True, 'workers': 8},
}
if WORKFLOW_SETTINGS['run']['smoke_mode']:
    E3SM_CASES = {'E3SM-FOSIRL': E3SM_CASES['E3SM-FOSIRL']}
    WORKFLOW_SETTINGS['run'].update(years=[1980, 1981], init_months=[11])
    WORKFLOW_SETTINGS['regrid'].update(target_dlat=5., target_dlon=5.)
    WORKFLOW_SETTINGS['dask']['workers'] = 2
    print('Archive-backed smoke mode: one E3SM case, CESM-SMYLE, two starts, one month.')
print('Field:', field, '| Years:', WORKFLOW_SETTINGS['run']['years'])
print('Metric:', WORKFLOW_SETTINGS['metric'])


## Inventory and cache plan

Inventory all requested initializations before starting expensive processing.
All E3SM members are required. Missing initializations cause an error rather than
silently shortening a case's comparison period. This also checks CESM-SMYLE
**monthly** benchmark availability and identifies the observation source.

Identity includes source file inventories, settings and adapter code. Compatible
metric caches can be reused without remapping the data. `require` needs source
metadata and existing compatible caches. Rerun this cell after changing settings.


In [ ]:
archive_plan = plan_archive_run(WORKFLOW_SETTINGS, E3SM_CASES, variable)
plan_table = pd.DataFrame([
    {'case': t['case'], 'init_month': t['month'], 'initializations': len(t['years']),
     'source_files': len(t['inventory']), 'action': 'prepare' if t['rebuild'] else 'reuse',
     'cache': t['path']}
    for t in archive_plan
])
display(plan_table)


## Dask resources

Start after the inventory succeeds. Rerunning this cell closes the previous cluster.
Run the cleanup cell when finished, including after an interrupted computation.


In [ ]:
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources

cluster, client, workflow_resources = restart_notebook_cluster(
    globals(),
    lambda: get_cluster_client(DaskConfig(
        cluster_type='local', workers=WORKFLOW_SETTINGS['dask']['workers']))
        if WORKFLOW_SETTINGS['dask']['enabled'] else (None, None),
)
if client is not None:
    display(client)


## Monthly preparation, common-grid calculation, and cache writing

For each missing case/month product, the module opens monthly archives, validates
represented verification months, converts units, regrids model and observation,
and computes the metric. Each source dataset is closed after the calculation.

To save space, the preparation and metric stages write **compact global block
indices, coverage and ratios**, not duplicate gridded monthly files. These live
under `<case>/initial_shock/metrics/atm/<field>/`, separate from ACC/RMSE outputs.
No climatology period is needed: scalar centering does not change standard deviation.


In [ ]:
comparison_by_month = compute_archive_plan(archive_plan, WORKFLOW_SETTINGS, variable)
for month, comparison in comparison_by_month.items():
    print(f'Initialization month {month:02d}')
    display(comparison[['std_ratio', 'model_std', 'observation_std',
                        'paired_sample_count', 'valid_ratio']])


## Coverage, comparison tables and heatmaps

Ratios above 1 indicate excess model variability; invalid ratios remain missing.
Check paired sample counts, spatial coverage, and observed standard deviation.
A small denominator can cause a large ratio. Different model ensemble sizes also
affect the variability of their ensemble means; no member-resampling adjustment
is made here. May and November comparisons are plotted separately.


In [ ]:
FIGURE_ROOT = Path(WORKFLOW_SETTINGS['paths']['figure_outdir'])
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
for month, comparison in comparison_by_month.items():
    # Hash suffix prevents different case selections/settings from overwriting figures.
    import hashlib
    identities = [t['digest'] for t in archive_plan if t['month'] == month]
    run_id = hashlib.sha256(''.join(identities).encode()).hexdigest()[:12]
    prefix = f'{field}_init{month:02d}_{run_id}'
    summary = comparison[['std_ratio', 'model_std', 'observation_std',
                          'paired_sample_count', 'valid_ratio']].to_dataframe().reset_index()
    display(summary)
    display(comparison[['model_area_fraction', 'observation_area_fraction']].min('block'))
    summary.to_csv(FIGURE_ROOT / f'{prefix}_summary.csv', index=False)
    fig = plot_std_ratio(comparison)
    fig.savefig(FIGURE_ROOT / f'{prefix}_std_ratio.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved comparison:', FIGURE_ROOT / f'{prefix}_std_ratio.png')


## Inspect one initialization

Choose a month and initialization year from the computed cohort. Model and observation
curves are independently centered within the window to show variability without the
mean bias. This does not change standard deviation; uncentered indices are cached.


In [ ]:
INSPECT_MONTH = WORKFLOW_SETTINGS['run']['init_months'][0]
INSPECT_YEAR = WORKFLOW_SETTINGS['run']['years'][0]
selected = comparison_by_month[INSPECT_MONTH].sel(Y=INSPECT_YEAR)
fig, ax = plt.subplots(figsize=(9, 4))
for case in selected.case.values:
    data = selected.sel(case=case)
    ax.plot(data.block, data.model_index - data.model_index.mean('block'),
            marker='o', label=f'{case}: model')
    ax.plot(data.block, data.observation_index - data.observation_index.mean('block'),
            linestyle='--', alpha=.6, label=f'{case}: observation')
ax.set(xlabel='Averaging block', ylabel=f"Window-centered global index ({variable['units']})",
       title=f'{field}: initialization {INSPECT_YEAR}-{INSPECT_MONTH:02d}')
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=.25)
fig.tight_layout()
plt.show()
plt.close(fig)


## Cleanup

Cached metrics remain available for plotting. Source files are closed inside the
workflow; this cell releases the optional distributed cluster.

See [methodology](../docs/initial_shock_std_index.md) for interpretation and
[the NCL reference](../temp/Compute_Std_Index_Initial_Shock_share.ncl).


In [ ]:
close_notebook_resources(globals())
print('Closed notebook Dask resources.')
